## Seq2seq translation with tatoeba dataset

## Data pre-processing

In [1]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import string
import re
import random
import os

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.autograd import Variable
from nltk.translate.bleu_score import sentence_bleu

In [2]:
import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.autograd import Variable

# Loss function: https://pytorch.org/docs/stable/nn.html#torch.nn.NLLLoss

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = torch.cuda.is_available()

In [ ]:
# %pip install gensim

In [3]:
import pandas as pd
from gensim.corpora.dictionary import Dictionary
from nltk import word_tokenize

#!pip install nltk  
import nltk
nltk.download('punkt')

/Users/glora/projects/populate-ns-lex/.venv-benepar/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[nltk_data] Downloading package punkt to /Users/glora/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
# !pip install benepar
# !pip install spacy
import spacy
import benepar
from nltk import Tree

In [5]:
print('import complete')

import complete


In [12]:
benepar.download("benepar_en3")

[nltk_data] Downloading package benepar_en3 to /usr/share/nltk_data...
[nltk_data]   Unzipping models/benepar_en3.zip.


True

In [6]:
from spacy.lang.en import English
import benepar

# 1) Create a blank English pipeline (only a tokenizer by default)
nlp = English()

# 2) Add a simple rule‐based sentencizer so benepar knows where your sentences are
nlp.add_pipe("sentencizer")

# 3) Make sure your benepar model is downloaded, then add it
#    (you only have to do the download once)
# benepar.download("benepar_en3")
nlp.add_pipe("benepar", config={"model": "benepar_en3"})

# 4) Check your pipeline
print(nlp.pipe_names)
# → ['sentencizer', 'benepar']

doc = nlp("This is a test sentence with a prepositional phrase in it.")
sents = list(doc.sents)
if sents:
    # benepar registers two main extensions on Span:
    #  - ._.parse_string  → the bracketed parse as a str
    #  - ._.constituency  → an nltk.Tree
    tree = sents[0]._.parse_string  
    print(tree)



/Users/glora/projects/populate-ns-lex/.venv-benepar/lib/python3.9/site-packages/benepar/parse_chart.py:169: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.

['sentencizer', 'benepar']
(S (NP (DT This)) (VP (VBZ is) (NP (NP (DT a) (NN test) (NN sentence)) (PP (IN with) (NP (NP (DT a) (JJ prepositional) (NN phrase)) (PP (IN in) (NP (PRP it))))))) (. .))


/Users/glora/projects/populate-ns-lex/.venv-benepar/lib/python3.9/site-packages/torch/distributions/distribution.py:55: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


In [7]:
from tqdm import tqdm_notebook as tqdm

In [18]:
import json
path_lexicon_pp = "../../pp_lexicon/master_pp_lexicon.json"
with open(path_lexicon_pp, "r", encoding="utf-8") as f:
    pp_lexicon = json.load(f)
    
len(pp_lexicon.items())

189

In [29]:
list_pp = list(pp_lexicon.keys())
# make sure your pp list is all lowercase
list_pp = {pp.lower() for pp in pp_lexicon.keys()}

def count_pp_lexicon(sent_str, pp_set):
    """Count how many prepositions (from pp_set) occur in the sentence."""
    doc = nlp(sent_str)
    return sum(1 for token in doc if token.text.lower() in pp_set)


In [ ]:
def unicodeToAscii(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s


def parse_sentence(sent_str):
    doc = nlp(sent_str)
    sent = next(doc.sents)              # first (and only) sentence
    return sent._.parse_string         # an parse tree in string

def count_pps(parse_str):
    # turn the bracketed string back into an nltk.Tree
    tree = Tree.fromstring(parse_str)
    # now count all PP nodes
    return sum(1 for t in tree.subtrees(lambda t: t.label() == "PP"))


def classify_complexity(tree):
    tree = Tree.fromstring(tree)
    # SBAR = subordinate clause → complex
    num_sbar = sum(1 for t in tree.subtrees(lambda t: t.label() == "SBAR"))
    # Count top‐level S children = coordinated clauses → compound
    top_s = sum(1 for child in tree if isinstance(child, Tree) and child.label() == "S")
    if num_sbar > 0:
        return "complex"
    elif top_s > 1:
        return "compound"
    else:
        return "simple"

pairs = []
input_path = './ind-eng/ind.txt'
with open(input_path, 'r', encoding='utf8') as fp:
    for line in fp:
        parts = line.strip().split('\t')
        if len(parts) < 2:
            continue
        eng, ind = parts[:2]
        if eng and ind:
            # append as a nested list
            pairs.append([eng, ind])
            
print(f"number of sentence pairs: {len(pairs)}")

number of pairs: 14881


In [55]:
from tqdm.auto import tqdm
# enable tqdm on pandas .apply
tqdm.pandas()

def process_pp_pairs(pairs, pp_set):
    """
    pairs: list of [english_str, indonesian_str]
    pp_set: set of lowercase prepositions (e.g. list_pp)
    returns: filtered & annotated DataFrame
    """
    # 1) build initial DF
    df = pd.DataFrame(pairs, columns=['English', 'Indonesian'])
    
    # 2) parse each English sentence
    df['ParseTree'] = df['English'].progress_apply(parse_sentence)
    
    # 3) drop where there are zero tree‐PPs
    df = df[df['ParseTree'].apply(count_pps) > 0].reset_index(drop=True)
    
    # 4) drop where lexicon lookup finds zero PPs
    df = df[df['English']
                .progress_apply(lambda s: count_pp_lexicon(s, pp_set)) > 0]\
           .reset_index(drop=True)
    
    # 5) classify complexity
    df['Complexity'] = df['ParseTree'].progress_apply(classify_complexity)
    
    # 6) normalize both sides
    df['English_normalized']    = df['English'].progress_apply(normalizeString)
    df['Indonesian_normalized'] = df['Indonesian'].progress_apply(normalizeString)
    
    return df

In [56]:
df_pp = process_pp_pairs(pairs, list_pp)
print(df_pp.shape)
df_pp.head()

  0%|          | 0/14881 [00:00<?, ?it/s]

/Users/glora/projects/populate-ns-lex/.venv-benepar/lib/python3.9/site-packages/torch/distributions/distribution.py:55: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


  0%|          | 0/4388 [00:00<?, ?it/s]

/Users/glora/projects/populate-ns-lex/.venv-benepar/lib/python3.9/site-packages/torch/distributions/distribution.py:55: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(


  0%|          | 0/4305 [00:00<?, ?it/s]

  0%|          | 0/4305 [00:00<?, ?it/s]

  0%|          | 0/4305 [00:00<?, ?it/s]

(4305, 6)


,English,Indonesian,ParseTree,Complexity,English_normalized,Indonesian_normalized
0,Go to bed.,Pergilah tidur.,(S (VP (VB Go) (PP (IN to) (NP (NN bed)))) (. .)),simple,go to bed .,pergilah tidur .
1,It's on me.,Aku yang traktir.,(S (NP (PRP It)) (VP (VBZ 's) (PP (IN on) (NP ...,simple,it s on me .,aku yang traktir .
2,Look at me.,Lihat saya.,(S (VP (VB Look) (PP (IN at) (NP (PRP me)))) (...,simple,look at me .,lihat saya .
3,Look at me.,Lihat aku.,(S (VP (VB Look) (PP (IN at) (NP (PRP me)))) (...,simple,look at me .,lihat aku .
4,Step on it!,Cepatlah!,(S (VP (VB Step) (PP (IN on) (NP (PRP it)))) (...,simple,step on it !,cepatlah !


In [63]:
df_pp['Complexity'].value_counts()

Complexity
simple      3763
complex      510
compound      32
Name: count, dtype: int64

In [57]:
df_pp.to_csv('tatoeba_pp_fix.csv')

In [58]:
print("Amount of tatoeba prepositional phrases:", df_pp.shape[0])

Amount of tatoeba prepositional phrases: 4305


In [59]:
MAX_LENGTH = 25
MIN_LENGTH = 4

In [60]:
df2 = df_pp.copy()

In [62]:
def should_keep_row(row):
    """ Should the current row be kept as training set"""
    # indo_num_words = len(word_tokenize(row["Indonesian"]))
    eng_num_words = len(word_tokenize(row["English_normalized"]))
    max_words_required = MAX_LENGTH - 2
    min_words_required = MIN_LENGTH

    return min_words_required <= eng_num_words <= max_words_required

df2["keep_row"] = df2.apply(should_keep_row, axis=1)
print(df2.shape)
df2.head()

print("Current shape: " + str(df2.shape))
df2 = df2[df2["keep_row"]]
print("New shape: " + str(df2.shape))
df2.head()
df2 = df2.reset_index().drop(columns=["keep_row"])
df2.head()

(4305, 7)
Current shape: (4305, 7)
New shape: (4298, 7)


,index,English,Indonesian,ParseTree,Complexity,English_normalized,Indonesian_normalized
0,0,Go to bed.,Pergilah tidur.,(S (VP (VB Go) (PP (IN to) (NP (NN bed)))) (. .)),simple,go to bed .,pergilah tidur .
1,1,It's on me.,Aku yang traktir.,(S (NP (PRP It)) (VP (VBZ 's) (PP (IN on) (NP ...,simple,it s on me .,aku yang traktir .
2,2,Look at me.,Lihat saya.,(S (VP (VB Look) (PP (IN at) (NP (PRP me)))) (...,simple,look at me .,lihat saya .
3,3,Look at me.,Lihat aku.,(S (VP (VB Look) (PP (IN at) (NP (PRP me)))) (...,simple,look at me .,lihat aku .
4,4,Step on it!,Cepatlah!,(S (VP (VB Step) (PP (IN on) (NP (PRP it)))) (...,simple,step on it !,cepatlah !


In [64]:
# Use a unique string to indicate START and END of a sentence.
# Assign a unique index to them.
START, START_IDX = '<s>',  0
END, END_IDX = '</s>', 1
UNK, UNK_IDX = 'UNK', 2

SOS_token = START_IDX
EOS_token = END_IDX

# added the START and the END symbol to the sentences. 
eng_tokens = df2['English_normalized'].str.lower().apply(word_tokenize).tolist()
english_sents = [[START] + tok + [END] for tok in eng_tokens]

indo_tokens = df2['Indonesian_normalized'].str.lower().apply(word_tokenize).tolist()
indo_sents = [[START] + tok + [END] for tok in indo_tokens]


# We're sort of getting into the data into the shape we want. 
# But now it's still too humanly readable and redundant.
## Cut-away: Computers like it to be simpler, more concise. -_-|||
print('First English sentence:', english_sents[0])
print('First Indo sentence:', indo_sents[0])

english_vocab = Dictionary([['<s>'], ['</s>'],['UNK']])
english_vocab.add_documents(english_sents)

indo_vocab = Dictionary([['<s>'], ['</s>'], ['UNK']])
indo_vocab.add_documents(indo_sents)

# First ten words in the vocabulary.
print('First 10 Indonesian words in Dictionary:\n', sorted(indo_vocab.items())[:10])
print()
print('First 10 English words in Dictionary:\n', sorted(english_vocab.items())[:10])

english_vocab = Dictionary([['<s>'], ['</s>'],['UNK']])
english_vocab.add_documents(english_sents)

indo_vocab = Dictionary([['<s>'], ['</s>'], ['UNK']])
indo_vocab.add_documents(indo_sents)

# First ten words in the vocabulary.
print('First 10 Indonesian words in Dictionary:\n', sorted(indo_vocab.items())[:10])
print()
print('First 10 English words in Dictionary:\n', sorted(english_vocab.items())[:10])

First English sentence: ['<s>', 'go', 'to', 'bed', '.', '</s>']
First Indo sentence: ['<s>', 'pergilah', 'tidur', '.', '</s>']
First 10 Indonesian words in Dictionary:
 [(0, '<s>'), (1, '</s>'), (2, 'UNK'), (3, '.'), (4, 'pergilah'), (5, 'tidur'), (6, 'aku'), (7, 'traktir'), (8, 'yang'), (9, 'lihat')]

First 10 English words in Dictionary:
 [(0, '<s>'), (1, '</s>'), (2, 'UNK'), (3, '.'), (4, 'bed'), (5, 'go'), (6, 'to'), (7, 'it'), (8, 'me'), (9, 'on')]
First 10 Indonesian words in Dictionary:
 [(0, '<s>'), (1, '</s>'), (2, 'UNK'), (3, '.'), (4, 'pergilah'), (5, 'tidur'), (6, 'aku'), (7, 'traktir'), (8, 'yang'), (9, 'lihat')]

First 10 English words in Dictionary:
 [(0, '<s>'), (1, '</s>'), (2, 'UNK'), (3, '.'), (4, 'bed'), (5, 'go'), (6, 'to'), (7, 'it'), (8, 'me'), (9, 'on')]


## Compute BLEU score

In [65]:
#input val_sent_pairs[0] english input to translate output is candidate
#val_sent_pairs[1] reference 
def calculate_bleu_score(reference_sent,candidate_sent):
    reference = [word_tokenize(reference_sent)]
    candidate = word_tokenize(candidate_sent)
    
    if '<s>' in candidate:
        candidate.remove('<s>')
    if '</s>' in candidate:
        candidate.remove('</s>')         
    gram_1_score = sentence_bleu(reference,candidate,weights=(1, 0, 0, 0))
    gram_2_score = sentence_bleu(reference,candidate,weights=(0.5, 0.5, 0, 0))
    gram_3_score = sentence_bleu(reference,candidate,weights=(0.33, 0.33, 0.33, 0))
    gram_4_score = sentence_bleu(reference,candidate,weights=(0.25, 0.25, 0.25, 0.25))
    blue_score = (gram_1_score+gram_2_score+gram_3_score+gram_4_score)/4
    #print(blue_score)
    return blue_score

## Utility methods

In [66]:
import pickle
# Lets save our dictionaries.
with open('tatoeba_pp_vocab_id-fix.Dictionary.pkl', 'wb') as fout:
   pickle.dump(indo_vocab, fout)
    
with open('tatoeba_pp_vocab_en-fix.Dictionary.pkl', 'wb') as fout:
   pickle.dump(english_vocab, fout)

In [ ]:
# #load vocab
# import pickle

# with open('dataset/tatoeba/tatoeba_pp_vocab_en-fix.Dictionary.pkl', 'rb') as fin:
#     indo_vocab1 = pickle.load(fin)

# with open('dataset/tatoeba/tatoeba_pp_vocab_en-fix.Dictionary.pkl', 'rb') as fin:
#     english_vocab1 = pickle.load(fin)

# print(type(indo_vocab1), len(indo_vocab1))
# print(type(english_vocab1), len(english_vocab1))

# # First ten words in the vocabulary.
# print('First 10 Indonesian words in Dictionary:\n', sorted(indo_vocab1.items())[:10])
# print()
# print('First 10 English words in Dictionary:\n', sorted(english_vocab1.items())[:10])


<class 'gensim.corpora.dictionary.Dictionary'> 3780
<class 'gensim.corpora.dictionary.Dictionary'> 3224
First 10 Indonesian words in Dictionary:
 [(0, '<s>'), (1, '</s>'), (2, 'UNK'), (3, '.'), (4, 'pergilah'), (5, 'tidur'), (6, 'aku'), (7, 'traktir'), (8, 'yang'), (9, 'lihat')]

First 10 English words in Dictionary:
 [(0, '<s>'), (1, '</s>'), (2, 'UNK'), (3, '.'), (4, 'bed'), (5, 'go'), (6, 'to'), (7, "'s"), (8, 'it'), (9, 'me')]


In [69]:
# Vectorizes a sentence with a given vocab
def vectorize_sent(sent, vocab):
    return vocab.doc2idx([START] + word_tokenize(sent.lower()) + [END], unknown_word_index=2)

# Creates a PyTorch variable from a sentence against a given vocab
def variable_from_sent(sent, vocab):
    vsent = vectorize_sent(sent, vocab)
    #print(vsent)
    result = Variable(torch.LongTensor(vsent).view(-1, 1))
    #print(result)
    return result.cuda() if use_cuda else result

# Test
new_kopi = "Go to bed"
variable_from_sent(new_kopi, english_vocab)

tensor([[0],
        [5],
        [6],
        [4],
        [1]])

## Split into train and validation

In [86]:
df_to_split = df2.copy()

In [87]:
from sklearn.model_selection import train_test_split


df_train, df_temp = train_test_split(
    df2,
    test_size=0.20,
    random_state=42,
    shuffle=True
)


df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,     
    random_state=42,
    shuffle=True
)


df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)


print(f"train:      {len(df_train)} rows ({len(df_train)/len(df2):.1%})")
print(f"validation: {len(df_val)} rows ({len(df_val)/len(df2):.1%})")
print(f"test:       {len(df_test)} rows ({len(df_test)/len(df2):.1%})")


train:      3438 rows (80.0%)
validation: 430 rows (10.0%)
test:       430 rows (10.0%)


In [92]:
df_test['Complexity'].value_counts()

Complexity
simple      380
complex      48
compound      2
Name: count, dtype: int64

In [99]:
df_train.head()

indo_tensors = df_train['Indonesian_normalized'].apply(lambda s: variable_from_sent(s, indo_vocab))
print(df_train.iloc[0]['Indonesian_normalized'])
df_train

english_tensors = df_train['English_normalized'].apply(lambda s: variable_from_sent(s, english_vocab))

sent_pairs = list(zip(english_tensors.values, indo_tensors.values)) ## training tensor pairs

pairs = list(zip(df_train['English_normalized'], df_train['Indonesian_normalized']))
print(f"training sentence pair")
print(pairs[8])

ini waktunya tidur .
training sentence pair
('tom got hit by a truck and died instantly .', 'tom ditabrak truk dan langsung meninggal .')


In [ ]:
def sent_tensor_pair(df_val_in):
    indo_val_tensors = df_val_in['Indonesian_normalized'].apply(lambda s: variable_from_sent(s, indo_vocab))
    english_val_tensors = df_val_in['English_normalized'].apply(lambda s: variable_from_sent(s, english_vocab))
    val_sent_tensor_pairs = list(zip(english_val_tensors.values, indo_val_tensors.values))
    val_sent_pairs = list(zip(df_val_in['English_normalized'], df_val_in['Indonesian_normalized']))
    return val_sent_pairs, val_sent_tensor_pairs


val_sent_pairs, val_sent_tensor_pairs = sent_tensor_pair(df_val)
test_sent_pairs, test_sent_tensor_pairs = sent_tensor_pair(df_test)


In [104]:
df_train['ParseTree'][2]

'(S (NP (PRP He)) (VP (VBD did) (RB not) (VP (VB have) (NP (NP (JJ much) (NN time)) (SBAR (S (VP (TO to) (VP (VB work) (PP (IN on) (NP (PRP$ his) (NN speech)))))))))) (. .))'

## Save train test split

In [110]:
df_train.to_csv('tatoeba_train.csv', index=False)
df_val.to_csv('tatoeba_val.csv', index=False)
df_test.to_csv('tatoeba_test.csv', index=False) 